In [2]:
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
import pandas as pd
import numpy as np
import math
import re
import warnings

In [3]:
# Making sure we are working within the same directory as the jupyter notebook (this will allow for poratbility of the project)
import os
## Verify we're in the right directory
print("Current directory:", os.getcwd())

Current directory: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting


# Process for ShariSheet

## Create DropDownMenu for Sheet `contract-load_courses`  from Sheet `DropDown`

In [5]:
import warnings
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

#wb = load_workbook("/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting/Step1_input_ShariSheets/ShariSheet_CHIS_FY2026_v20260212.xlsx")
wb = load_workbook("/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting/0_source_files/Step01_ShariSheet_template_v20260212.xlsx")

#wb = load_workbook("Step2_input_SCHEDULES_PAST_year_filled/SEM_budget_FY25_FY26.xlsx")
ws_main = wb["contract-load_courses"]
ws_drop = wb["DropDown"]

# --- STEP 2: Create DropDownMenu for Sheet1 from Sheet "DropDown" ---


# Find max rows in DropDown
max_row = ws_drop.max_row

# Build dictionary: header -> (column_letter, start_row, end_row)
dropdown_sources = {}

for col in ws_drop.iter_cols(min_row=1, max_row=1):
    header = col[0].value
    if header:
        col_letter = col[0].column_letter

        # find last non-empty cell in that column
        end_row = max_row
        while end_row > 1 and not ws_drop[f"{col_letter}{end_row}"].value:
            end_row -= 1

        # Only add if there are values
        if end_row > 1:
            dropdown_sources[header] = f"=DropDown!${col_letter}$2:${col_letter}${end_row}"

# Apply dropdowns to Sheet1
for col in ws_main.iter_cols(min_row=1, max_row=1):
    header = col[0].value
    if header in dropdown_sources:
        formula = dropdown_sources[header]
        dv = DataValidation(type="list", formula1=formula, allow_blank=True)

        col_letter = col[0].column_letter
        dv.add(f"{col_letter}2:{col_letter}5000")

        ws_main.add_data_validation(dv)


# --- STEP 3: Freeze Panes + AutoFilter ---

ws_main.freeze_panes = "B2"              # Freeze first row + first column
ws_main.auto_filter.ref = ws_main.dimensions  # AutoFilter on full header row

#wb.save("/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting/Step1_input_ShariSheets/ShariSheet_CHIS_FY2026_v20260212.xlsx")
wb.save("/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting/0_source_files/Step01_ShariSheet_template_v20260212.xlsx")




## Bulk updating multiple ShariSheets

In [6]:
import os
import glob
import warnings
from copy import copy

from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# folder containing ShariSheets that need to get updated DropDown menus
FOLDER = "/Users/oliverglanz/Desktop/sharisheetsfy25/"

# template file
TEMPLATE_FILE = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartBudgeting/0_source_files/Step01_ShariSheet_template_v20260212.xlsx"

DROPDOWN_SHEET_NAME = "DropDown"
TARGET_SHEET_NAME = "contract-load_courses"

MAX_APPLY_ROWS = 5000  # dropdowns applied to row 2..MAX_APPLY_ROWS


def copy_sheet_contents(src_ws, dst_ws):
    """
    Copies values + styles + dimensions + merges from src_ws to dst_ws.
    (Does NOT copy charts/images; usually fine for dropdown list sheets.)
    """
    # Clear existing cells in destination
    if dst_ws.max_row > 1 or dst_ws.max_column > 1:
        dst_ws.delete_rows(1, dst_ws.max_row)

    # Copy column dimensions (width, hidden, etc.)
    for col_letter, dim in src_ws.column_dimensions.items():
        dst_ws.column_dimensions[col_letter].width = dim.width
        dst_ws.column_dimensions[col_letter].hidden = dim.hidden
        dst_ws.column_dimensions[col_letter].outline_level = dim.outline_level
        dst_ws.column_dimensions[col_letter].collapsed = dim.collapsed

    # Copy row dimensions (height, hidden, etc.)
    for row_idx, dim in src_ws.row_dimensions.items():
        dst_ws.row_dimensions[row_idx].height = dim.height
        dst_ws.row_dimensions[row_idx].hidden = dim.hidden
        dst_ws.row_dimensions[row_idx].outline_level = dim.outline_level
        dst_ws.row_dimensions[row_idx].collapsed = dim.collapsed

    # Copy cells (values + styles)
    for row in src_ws.iter_rows():
        for cell in row:
            dst_cell = dst_ws.cell(row=cell.row, column=cell.column, value=cell.value)

            if cell.has_style:
                dst_cell._style = copy(cell._style)
            dst_cell.number_format = cell.number_format
            dst_cell.font = copy(cell.font)
            dst_cell.fill = copy(cell.fill)
            dst_cell.border = copy(cell.border)
            dst_cell.alignment = copy(cell.alignment)
            dst_cell.protection = copy(cell.protection)
            dst_cell.comment = copy(cell.comment) if cell.comment else None

    # Copy merged cells
    for merged_range in list(src_ws.merged_cells.ranges):
        dst_ws.merge_cells(str(merged_range))

    # Copy sheet properties that are often relevant
    dst_ws.sheet_format = copy(src_ws.sheet_format)
    dst_ws.sheet_properties = copy(src_ws.sheet_properties)
    dst_ws.page_margins = copy(src_ws.page_margins)
    dst_ws.page_setup = copy(src_ws.page_setup)
    dst_ws.print_options = copy(src_ws.print_options)
    dst_ws.views = copy(src_ws.views)


def rebuild_dropdown_sources(ws_drop):
    """
    Build dict: header -> formula range, based on first-row headers in ws_drop.
    """
    max_row = ws_drop.max_row
    dropdown_sources = {}

    for col in ws_drop.iter_cols(min_row=1, max_row=1):
        header = col[0].value
        if not header:
            continue

        col_letter = col[0].column_letter

        # find last non-empty cell in that column
        end_row = max_row
        while end_row > 1 and not ws_drop[f"{col_letter}{end_row}"].value:
            end_row -= 1

        if end_row > 1:
            dropdown_sources[header] = f"=DropDown!${col_letter}$2:${col_letter}${end_row}"

    return dropdown_sources


def clear_existing_validations(ws):
    """
    Remove all existing data validations from a worksheet.
    """
    if getattr(ws, "data_validations", None) is not None:
        ws.data_validations.dataValidation = []


def apply_dropdowns(ws_main, dropdown_sources, max_apply_rows=5000):
    """
    Apply dropdown validations to ws_main based on matching headers in row 1.
    """
    clear_existing_validations(ws_main)

    for col in ws_main.iter_cols(min_row=1, max_row=1):
        header = col[0].value
        if header in dropdown_sources:
            formula = dropdown_sources[header]
            dv = DataValidation(type="list", formula1=formula, allow_blank=True)

            col_letter = col[0].column_letter
            dv.add(f"{col_letter}2:{col_letter}{max_apply_rows}")

            ws_main.add_data_validation(dv)

    # Freeze + filter (same as your original intent)
    ws_main.freeze_panes = "B2"
    ws_main.auto_filter.ref = ws_main.dimensions


def process_workbook(path, template_ws_drop):
    wb = load_workbook(path, data_only=False)

    # Ensure DropDown exists
    if DROPDOWN_SHEET_NAME in wb.sheetnames:
        ws_drop = wb[DROPDOWN_SHEET_NAME]
    else:
        ws_drop = wb.create_sheet(DROPDOWN_SHEET_NAME)

    # Overwrite DropDown contents from template
    copy_sheet_contents(template_ws_drop, ws_drop)

    # Rebuild dropdown sources from the (new) DropDown
    dropdown_sources = rebuild_dropdown_sources(ws_drop)

    # Apply to contract-load_courses if present
    if TARGET_SHEET_NAME in wb.sheetnames:
        ws_main = wb[TARGET_SHEET_NAME]
        apply_dropdowns(ws_main, dropdown_sources, MAX_APPLY_ROWS)
        status = "UPDATED"
    else:
        status = f"SKIPPED (missing sheet '{TARGET_SHEET_NAME}')"

    wb.save(path)
    return status


def main():
    # Load template once
    wb_t = load_workbook(TEMPLATE_FILE, data_only=False)
    if DROPDOWN_SHEET_NAME not in wb_t.sheetnames:
        raise ValueError(f"Template is missing sheet '{DROPDOWN_SHEET_NAME}': {TEMPLATE_FILE}")
    template_ws_drop = wb_t[DROPDOWN_SHEET_NAME]

    # Find all xlsx files in folder
    files = sorted(glob.glob(os.path.join(FOLDER, "*.xlsx")))

    if not files:
        raise FileNotFoundError(f"No .xlsx files found in: {FOLDER}")

    for f in files:
        # Optional: skip the template if it happens to be in the same folder
        if os.path.abspath(f) == os.path.abspath(TEMPLATE_FILE):
            continue

        try:
            status = process_workbook(f, template_ws_drop)
            print(f"{status}: {os.path.basename(f)}")
        except Exception as e:
            print(f"ERROR: {os.path.basename(f)} -> {e}")


if __name__ == "__main__":
    main()


UPDATED: ShariSheet_CHIS_FY2025_v20260127.xlsx
UPDATED: ShariSheet_DMIN_FY2025_v20260127.xlsx
UPDATED: ShariSheet_DSLE_FY2025_v20260127.xlsx
UPDATED: ShariSheet_MSSN_FY2025_v20260127.xlsx
UPDATED: ShariSheet_NTST_FY2025_v20260127.xlsx
UPDATED: ShariSheet_OTST_FY2025_v20260127.xlsx
UPDATED: ShariSheet_PATH_FY2025_v20260127.xlsx
UPDATED: ShariSheet_THST_FY2025_v20260127.xlsx
